# PyTorch — Klasifikasi dengan Kelas Timpang (Imbalanced)

**Kapan pakai file ini:** satu kelas jauh lebih banyak (rasio 5:1 ke atas). Contoh: deteksi keluhan, fraud.

Versi PyTorch dari `cheatsheet-imbalanced.ipynb`.
Alur dan CONFIG dibuat semirip mungkin — yang berbeda hanya bagian model: tidak ada
`TfidfVectorizer` dan `.fit()`, melainkan **vocab → tensor → arsitektur → training loop manual**.

**Evaluasinya tetap memakai scikit-learn** (`classification_report`, `confusion_matrix`,
`f1_score`) karena metriknya sama saja, dan §8 menghitung pembanding TF-IDF supaya angkanya jujur.

```
intip file -> ambil kolom -> tokenisasi -> vocab+tensor -> model -> training loop
-> evaluasi (sklearn) -> prediksi -> output
```

Teori: `cheatsheet_pytorch.ipynb` · Semua opsi: `latihan_pytorch.ipynb`

---
## §1 · CONFIG — ubah di sini saja

In [1]:
# ---------------- DATA: struktur file ----------------
PATH   = "data/keluhan_imbalanced.csv"
SEP    = None
HEADER = "infer"
ENC    = None

# ---------------- DATA: kolom ----------------
TEXT_COL   = None
LABEL_COL  = None
LABEL_MAP  = None
MULTILABEL = False
SAMPLE     = None

BAHASA = "en"

# ---------------- PREPROCESSING (True/False) ----------------
LOWERCASE   = True
MASK        = True      # URL/mention/angka -> token generik
STOPWORD    = False     # untuk neural sering justru DIBIARKAN: urutan kata ikut informatif
JAGA_NEGASI = True

# ---------------- TEKS -> TENSOR ----------------
MIN_FREQ  = 2           # kata dengan frekuensi < ini jadi <unk>
MAX_VOCAB = 20000
MAX_LEN   = 48          # token per dokumen, sisanya dipotong

# ---------------- ARSITEKTUR ----------------
ARSITEKTUR    = "meanpool"   # meanpool | cnn | lstm | gru
DIM           = 100     # dimensi embedding          (semua arsitektur)
HIDDEN        = 128     # unit LSTM/GRU              (lstm, gru)
N_FILTER      = 100     # filter per ukuran kernel   (cnn)
KERNEL        = (3, 4, 5)                          # (cnn)
BIDIRECTIONAL = True    # dua arah                   (lstm, gru)
DROPOUT       = 0.3

# ---------------- TRAINING ----------------
EPOCHS       = 10
BATCH        = 64
LR           = 1e-3
WEIGHT_DECAY = 0.0
SEIMBANGKAN  = True

TEST_SIZE, VAL_SIZE, SEED = 0.2, 0.15, 42

CONTOH_BARU = ["my order never arrived and support is ignoring me",
               "just got home, going to watch a movie tonight"]

---
## §2 · Intip struktur file

Jalankan sebelum apa pun. Kalau nama kolom / separator / encoding tidak sesuai dugaan,
perbaiki di CONFIG lalu ulangi sel ini.

In [2]:
import re, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score


def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

# ---- lihat 3 baris pertama file MENTAH ----
print("\n=== isi mentah 3 baris pertama ===")
with open(PATH, encoding="utf-8", errors="replace") as f:
    for i, baris in zip(range(3), f):
        print(f"  {i}: {baris.rstrip()[:110]}")

_sep = SEP or ("\t" if PATH.endswith((".tsv", ".tab")) else ",")
for _enc in ([ENC] if ENC else ["utf-8", "latin-1", "cp1252"]):
    try:
        raw = pd.read_csv(PATH, sep=_sep, header=HEADER, encoding=_enc,
                          engine="python", on_bad_lines="skip")
        break
    except (UnicodeDecodeError, UnicodeError):
        continue

print(f"\n=== terbaca: {raw.shape[0]} baris x {raw.shape[1]} kolom "
      f"(sep={_sep!r}, encoding={_enc}) ===")
print("nama kolom :", list(raw.columns))
print("\njumlah nilai unik per kolom:")
print(raw.nunique().to_string())

_teks = [c for c in raw.columns if raw[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
_tebak_teks = max(_teks, key=lambda c: raw[c].astype(str).str.len().mean()) if _teks else None
_kand = [(raw[c].nunique(), c) for c in raw.columns
         if c != _tebak_teks and 2 <= raw[c].nunique() <= 200]
print("\n=== saran untuk CONFIG ===")
print(f"TEXT_COL   = {_tebak_teks!r}")
print(f"LABEL_COL  = {min(_kand)[1]!r}" if _kand else "LABEL_COL  = ?")

torch 2.6.0+cu124 | device: cuda | NVIDIA GeForce RTX 3060 Laptop GPU

=== isi mentah 3 baris pertama ===
  0: text,label
  1: Hes off ,netral
  2: "Heading to aledo, tx to meet the tour team and head to san angelo for the next event. Im going to miss my car

=== terbaca: 1777 baris x 2 kolom (sep=',', encoding=utf-8) ===
nama kolom : ['text', 'label']

jumlah nilai unik per kolom:
text     1777
label       2

=== saran untuk CONFIG ===
TEXT_COL   = 'text'
LABEL_COL  = 'label'


---
## §3 · Ambil kolom teks & label

In [3]:
text_col, label_col = TEXT_COL, LABEL_COL
if text_col is None:
    kand = [c for c in raw.columns if raw[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
    text_col = max(kand, key=lambda c: raw[c].astype(str).str.len().mean())
if label_col is None:
    batas = 200 if MULTILABEL else 20
    kand = [(raw[c].nunique(), c) for c in raw.columns
            if c != text_col and 2 <= raw[c].nunique() <= batas]
    label_col = min(kand)[1]
print(f"kolom teks: {text_col!r} | kolom label: {label_col!r}")

df = raw.rename(columns={text_col: "text", label_col: "label"})[["text", "label"]].copy()
df["text"] = df["text"].astype(str).str.strip()
if not MULTILABEL:
    df["label"] = df["label"].apply(lambda v: v.strip().lower() if isinstance(v, str) else v)
if LABEL_MAP:
    df["label"] = df["label"].map(LABEL_MAP).fillna(df["label"])

n0 = len(df)
df = df[df["text"].str.len() >= 3]
df = df[~df["text"].str.lower().isin({"nan", "none", "na", "-"})]
df = df.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).reset_index(drop=True)
if SAMPLE and SAMPLE < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE, random_state=SEED,
                             stratify=None if MULTILABEL else df["label"])
    df = df.reset_index(drop=True)
print(f"setelah dibersihkan: {n0} -> {len(df)} baris")

kolom teks: 'text' | kolom label: 'label'
setelah dibersihkan: 1777 -> 1777 baris


In [4]:
KELAS = sorted(set(map(str, df["label"])))
l2i = {k: i for i, k in enumerate(KELAS)}
Y = np.array([l2i[str(v)] for v in df["label"]])
N_OUT = len(KELAS)

print("kelas     :", KELAS)
print("distribusi:", df["label"].value_counts().to_dict())

kelas     : ['keluhan', 'netral']
distribusi: {'netral': 1600, 'keluhan': 177}


---
## §4 · Tokenisasi

Untuk jalur neural, teks tidak diubah jadi TF-IDF melainkan jadi **daftar token**, lalu token
diganti indeks angka di §5. Stopword sering justru **tidak** dibuang untuk neural, karena urutan
dan kata fungsi ikut membawa informasi bagi CNN/LSTM.

In [5]:
STOP_EN = {"i","me","my","we","you","your","he","she","it","they","them","this","that","is","are",
           "was","were","be","the","a","an","and","but","or","of","at","by","for","with","to",
           "from","in","on","so","than","too","very","just","now","have","has","had","do","did"}
STOP_ID = {"yang","dan","di","ke","dari","ini","itu","untuk","dengan","pada","adalah","ada","saya",
           "kamu","dia","kami","kita","mereka","akan","sudah","juga","atau","karena","saja"}
NEGASI  = {"no","not","never","nor","cannot"} | {"tidak","bukan","tanpa","jangan","belum","kurang"}

stop = (STOP_EN if BAHASA == "en" else STOP_ID)
if JAGA_NEGASI:
    stop = stop - NEGASI


def tokenize(teks):
    t = str(teks)
    if LOWERCASE:
        t = t.lower()
    if MASK:
        t = re.sub(r"http\S+|www\.\S+|\b\S+\.(?:com|org|net|ly|id|co)\S*", " urltoken ", t)
        t = re.sub(r"@\w+", " usertoken ", t)
        t = re.sub(r"\b\d+\b", " numtoken ", t)
    kata = re.findall(r"[a-zA-Z]+", t)
    if STOPWORD:
        kata = [w for w in kata if w not in stop]
    return kata or ["kosongtoken"]


print("contoh token:", tokenize(df["text"].iloc[0])[:12])

contoh token: ['hes', 'off']


---
## §5 · Vocab → tensor → DataLoader

Empat langkah yang menggantikan satu baris `TfidfVectorizer`: bangun **vocab** (hanya dari data
latih), ubah token jadi **indeks**, **padding** supaya satu batch sama panjang, lalu bungkus di
`DataLoader`. Panjang asli tiap kalimat disimpan karena dipakai LSTM dan mean pooling.

In [6]:
from collections import Counter

PAD, UNK = 0, 1

# --- split dulu, baru bangun vocab: vocab HANYA boleh dari data latih ---
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, random_state=SEED,
                                  stratify=None if MULTILABEL else df["label"])
idx_tr, idx_val = train_test_split(idx_tr, test_size=VAL_SIZE, random_state=SEED,
                                   stratify=None if MULTILABEL else df["label"].iloc[idx_tr])
print("train:", len(idx_tr), "| val:", len(idx_val), "| test:", len(idx_te))

c = Counter(w for t in df["text"].iloc[idx_tr] for w in tokenize(t))
itos = ["<pad>", "<unk>"] + [w for w, n in c.most_common(MAX_VOCAB) if n >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
print("ukuran vocab:", len(itos))


def encode(teks):
    return [stoi.get(w, UNK) for w in tokenize(teks)][:MAX_LEN] or [UNK]


class DatasetTeks(Dataset):
    def __init__(self, idx):
        self.ids = [encode(t) for t in df["text"].iloc[idx]]
        self.y = Y[idx]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return torch.tensor(self.ids[i], dtype=torch.long), self.y[i]


MIN_LEN = max(KERNEL) if ARSITEKTUR == "cnn" else 1


def collate(batch):
    urut, label = zip(*batch)
    panjang = torch.tensor([len(x) for x in urut], dtype=torch.long)
    padded = pad_sequence(urut, batch_first=True, padding_value=PAD)
    if padded.size(1) < MIN_LEN:
        padded = F.pad(padded, (0, MIN_LEN - padded.size(1)), value=PAD)
    y = torch.tensor(np.array(label), dtype=torch.float if MULTILABEL else torch.long)
    return padded, panjang, y


mk = lambda idx, sh: DataLoader(DatasetTeks(idx), batch_size=BATCH, shuffle=sh, collate_fn=collate)
dl_tr, dl_val, dl_te = mk(idx_tr, True), mk(idx_val, False), mk(idx_te, False)

xb, lb, yb = next(iter(dl_tr))
print("bentuk batch:", tuple(xb.shape), "-> (batch, panjang terpanjang di batch)")

train: 1207 | val: 214 | test: 356
ukuran vocab: 1277
bentuk batch: (64, 32) -> (batch, panjang terpanjang di batch)


---
## §6 · Model — pilih arsitektur lewat `ARSITEKTUR` di CONFIG

In [7]:
class MeanPool(nn.Module):
    def __init__(self, n_vocab, n_out):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, DIM, padding_idx=PAD)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(DIM, n_out)

    def forward(self, x, panjang):
        e = self.emb(x)
        mask = (x != PAD).unsqueeze(-1).float()
        return self.fc(self.drop((e * mask).sum(1) / mask.sum(1).clamp(min=1)))


class CNNTeks(nn.Module):                      # Kim 2014, slide 02b hal. 10
    def __init__(self, n_vocab, n_out):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, DIM, padding_idx=PAD)
        self.convs = nn.ModuleList([nn.Conv1d(DIM, N_FILTER, k) for k in KERNEL])
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(N_FILTER * len(KERNEL), n_out)

    def forward(self, x, panjang=None):
        e = self.emb(x).transpose(1, 2)
        f = [F.relu(conv(e)).max(dim=2).values for conv in self.convs]
        return self.fc(self.drop(torch.cat(f, dim=1)))


class RNNTeks(nn.Module):                      # LSTM/GRU, slide 02b hal. 11
    def __init__(self, n_vocab, n_out):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, DIM, padding_idx=PAD)
        kelas = nn.LSTM if ARSITEKTUR == "lstm" else nn.GRU
        self.rnn = kelas(DIM, HIDDEN, batch_first=True, bidirectional=BIDIRECTIONAL)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN * (2 if BIDIRECTIONAL else 1), n_out)

    def forward(self, x, panjang):
        packed = pack_padded_sequence(self.emb(x), panjang.cpu(), batch_first=True,
                                      enforce_sorted=False)
        keluar = self.rnn(packed)[1]
        h = keluar[0] if isinstance(keluar, tuple) else keluar
        h = torch.cat([h[-2], h[-1]], dim=1) if BIDIRECTIONAL else h[-1]
        return self.fc(self.drop(h))


set_seed()
model = {"meanpool": MeanPool, "cnn": CNNTeks,
         "lstm": RNNTeks, "gru": RNNTeks}[ARSITEKTUR](len(itos), N_OUT).to(device)
print(f"arsitektur : {ARSITEKTUR}")
print(f"parameter  : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

arsitektur : meanpool
parameter  : 127,902


---
## §7 · Loss + training loop

Lima baris di dalam loop, urutannya tidak boleh salah:
`zero_grad → forward → loss → backward → step`. Bobot terbaik disimpan menurut **skor validasi**,
bukan epoch terakhir — kalau tidak, yang kamu evaluasi adalah model yang sudah mulai overfit.

In [8]:
# Bobot kelas dihitung dari frekuensi di data LATIH.
# Ini padanan class_weight="balanced" milik sklearn.
if SEIMBANGKAN:
    n = np.bincount(Y[idx_tr], minlength=N_OUT)
    bobot = torch.tensor(len(idx_tr) / (N_OUT * np.maximum(n, 1)),
                         dtype=torch.float, device=device)
    print("jumlah per kelas:", dict(zip(KELAS, n.tolist())))
    print("bobot loss      :", dict(zip(KELAS, bobot.cpu().numpy().round(2).tolist())))
    criterion = nn.CrossEntropyLoss(weight=bobot)
else:
    criterion = nn.CrossEntropyLoss()
    print("loss: CrossEntropyLoss tanpa bobot")

jumlah per kelas: {'keluhan': 121, 'netral': 1086}
bobot loss      : {'keluhan': 4.989999771118164, 'netral': 0.5600000023841858}


In [9]:
@torch.no_grad()
def prediksi(loader):
    model.eval()
    P, T = [], []
    for x, panjang, y in loader:
        logits = model(x.to(device), panjang.to(device))
        P.append((torch.sigmoid(logits) > 0.5).int().cpu() if MULTILABEL
                 else logits.argmax(1).cpu())
        T.append(y.cpu())
    return torch.cat(P).numpy(), torch.cat(T).numpy()


optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
rata = "micro" if MULTILABEL else "macro"

terbaik, state = -1, None
for ep in range(1, EPOCHS + 1):
    model.train()
    total, t0 = 0.0, time.time()
    for x, panjang, y in dl_tr:
        x, panjang, y = x.to(device), panjang.to(device), y.to(device)
        optimizer.zero_grad()                       # 1. hapus gradien batch sebelumnya
        loss = criterion(model(x, panjang), y)      # 2. forward + hitung loss
        loss.backward()                             # 3. hitung gradien
        optimizer.step()                            # 4. perbarui bobot
        total += loss.item() * y.size(0)
    pv, tv = prediksi(dl_val)
    f1v = f1_score(tv, pv, average=rata, zero_division=0)
    if f1v > terbaik:                               # simpan bobot terbaik menurut VALIDASI
        terbaik, state = f1v, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"  epoch {ep:2d}  loss {total/len(dl_tr.dataset):.4f}  val_f1 {f1v:.4f}"
          f"  ({time.time()-t0:.1f}s)")

model.load_state_dict(state)
print(f"\nbobot terbaik dikembalikan (val_f1 {terbaik:.4f})")

  epoch  1  loss 0.7053  val_f1 0.4133  (0.3s)
  epoch  2  loss 0.6863  val_f1 0.4527  (0.1s)
  epoch  3  loss 0.6756  val_f1 0.4467  (0.1s)
  epoch  4  loss 0.6626  val_f1 0.4406  (0.1s)


  epoch  5  loss 0.6430  val_f1 0.4467  (0.1s)
  epoch  6  loss 0.6274  val_f1 0.4618  (0.1s)
  epoch  7  loss 0.6304  val_f1 0.4919  (0.0s)
  epoch  8  loss 0.6197  val_f1 0.4911  (0.1s)


  epoch  9  loss 0.6110  val_f1 0.4911  (0.0s)
  epoch 10  loss 0.5935  val_f1 0.5001  (0.0s)

bobot terbaik dikembalikan (val_f1 0.5001)


---
## §8 · Evaluasi (pakai scikit-learn) + pembanding TF-IDF

In [10]:
pred, ytrue = prediksi(dl_te)
hitung = np.bincount(Y[idx_tr], minlength=N_OUT)
i_min, i_maj = int(hitung.argmin()), int(hitung.argmax())

print("akurasi :", round(accuracy_score(ytrue, pred), 3), " <- JANGAN dijadikan patokan")
print("f1 macro:", round(f1_score(ytrue, pred, average="macro"), 3))
print(f"f1 kelas '{KELAS[i_min]}' (minoritas):",
      round(f1_score(ytrue, pred, labels=[i_min], average="macro", zero_division=0), 3),
      " <- INI yang penting\n")
print(classification_report(ytrue, pred, target_names=KELAS, zero_division=0))

print("Confusion matrix (baris = aktual, kolom = prediksi):")
print(pd.DataFrame(confusion_matrix(ytrue, pred),
                   index=["true_" + k for k in KELAS],
                   columns=["pred_" + k for k in KELAS]).to_string())

print(f"\nModel yang SELALU menebak '{KELAS[i_maj]}' dapat akurasi "
      f"{(ytrue == i_maj).mean():.3f} dengan recall minoritas 0.")

# ---- pembanding: TF-IDF + LogisticRegression pada data yang sama ----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

i_tr = np.concatenate([idx_tr, idx_val])
klasik = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
                   ("clf", LogisticRegression(max_iter=1000,
                                              class_weight="balanced" if SEIMBANGKAN else None))])
klasik.fit(df["text"].iloc[i_tr], Y[i_tr])
f1_klasik = f1_score(ytrue, klasik.predict(df["text"].iloc[idx_te]),
                     average="macro", zero_division=0)
f1_neural = f1_score(ytrue, pred, average="macro")

print(f"\nPyTorch ({ARSITEKTUR:9s}) f1_macro = {f1_neural:.3f}")
print(f"TF-IDF + LogReg      f1_macro = {f1_klasik:.3f}")
print("->", "neural menang" if f1_neural > f1_klasik
      else "model klasik menang -- wajar untuk data sebesar ini")

akurasi : 0.694  <- JANGAN dijadikan patokan
f1 macro: 0.563
f1 kelas 'keluhan' (minoritas): 0.323  <- INI yang penting

              precision    recall  f1-score   support

     keluhan       0.21      0.74      0.32        35
      netral       0.96      0.69      0.80       321

    accuracy                           0.69       356
   macro avg       0.58      0.72      0.56       356
weighted avg       0.89      0.69      0.76       356

Confusion matrix (baris = aktual, kolom = prediksi):
              pred_keluhan  pred_netral
true_keluhan            26            9
true_netral            100          221

Model yang SELALU menebak 'netral' dapat akurasi 0.902 dengan recall minoritas 0.



PyTorch (meanpool ) f1_macro = 0.563
TF-IDF + LogReg      f1_macro = 0.595
-> model klasik menang -- wajar untuk data sebesar ini


---
## §9 · Prediksi teks baru

In [11]:
@torch.no_grad()
def prediksi_teks(daftar):
    model.eval()
    ids = [torch.tensor(encode(t)) for t in daftar]
    panjang = torch.tensor([len(i) for i in ids])
    x = pad_sequence(ids, batch_first=True, padding_value=PAD)
    if x.size(1) < MIN_LEN:
        x = F.pad(x, (0, MIN_LEN - x.size(1)), value=PAD)
    prob = F.softmax(model(x.to(device), panjang.to(device)), dim=1).cpu().numpy()
    return [(t, KELAS[int(p.argmax())], float(p.max())) for t, p in zip(daftar, prob)]


for teks, label, yakin in prediksi_teks(CONTOH_BARU):
    print(f"  {label:10s} ({yakin:.0%})  <- {teks[:58]}")

  keluhan    (59%)  <- my order never arrived and support is ignoring me
  netral     (58%)  <- just got home, going to watch a movie tonight


---
## §10 · Output — simpan model & tulis hasil

Untuk PyTorch yang disimpan adalah **`state_dict`**, bukan objek modelnya. Vocab (`itos`) dan
daftar kelas ikut disimpan — tanpa itu model tidak bisa dipakai lagi karena tidak tahu kata mana
berindeks berapa.

In [12]:
import json

NAMA = PATH.split("/")[-1].split(".")[0]
torch.save({"state_dict": model.state_dict(), "itos": itos, "kelas": KELAS,
            "cfg": dict(ARSITEKTUR=ARSITEKTUR, DIM=DIM, HIDDEN=HIDDEN, N_FILTER=N_FILTER,
                        KERNEL=KERNEL, BIDIRECTIONAL=BIDIRECTIONAL, DROPOUT=DROPOUT,
                        MAX_LEN=MAX_LEN, BAHASA=BAHASA, LOWERCASE=LOWERCASE, MASK=MASK,
                        STOPWORD=STOPWORD, JAGA_NEGASI=JAGA_NEGASI)},
           f"model_pytorch_{NAMA}.pt")

hasil = pd.DataFrame({"text": df["text"].iloc[idx_te].values,
                      "aktual":   [KELAS[i] for i in ytrue],
                      "prediksi": [KELAS[i] for i in pred]})
hasil["benar"] = hasil["aktual"] == hasil["prediksi"]
hasil.to_csv(f"hasil_pytorch_{NAMA}.csv", index=False)

ringkas = {"dataset": PATH, "arsitektur": ARSITEKTUR, "embedding_dim": DIM,
           "vocab": len(itos), "epochs": EPOCHS, "device": str(device),
           "n_train": len(idx_tr), "n_val": len(idx_val), "n_test": len(idx_te),
           "akurasi": round(float(accuracy_score(ytrue, pred)), 4),
           "f1_macro": round(float(f1_score(ytrue, pred, average="macro")), 4),
           "f1_val_terbaik": round(float(terbaik), 4)}
with open(f"ringkasan_pytorch_{NAMA}.json", "w") as f:
    json.dump(ringkas, f, indent=2)

print("tersimpan:")
print(f"  model_pytorch_{NAMA}.pt        <- state_dict + vocab + kelas")
print(f"  hasil_pytorch_{NAMA}.csv       <- {len(hasil)} baris")
print(f"  ringkasan_pytorch_{NAMA}.json")
print()
print(json.dumps(ringkas, indent=2))

# ---- cek: model yang dimuat ulang memberi hasil yang sama ----
ckpt = torch.load(f"model_pytorch_{NAMA}.pt", map_location=device, weights_only=False)
model2 = {"meanpool": MeanPool, "cnn": CNNTeks,
          "lstm": RNNTeks, "gru": RNNTeks}[ARSITEKTUR](len(ckpt["itos"]), N_OUT).to(device)
model2.load_state_dict(ckpt["state_dict"])
model2.eval()
with torch.no_grad():
    cek = torch.cat([model2(x.to(device), l.to(device)).argmax(1).cpu()
                     for x, l, _ in dl_te]).numpy()
print("\nmuat ulang cocok:", bool((cek == pred).all()))

tersimpan:
  model_pytorch_keluhan_imbalanced.pt        <- state_dict + vocab + kelas
  hasil_pytorch_keluhan_imbalanced.csv       <- 356 baris
  ringkasan_pytorch_keluhan_imbalanced.json

{
  "dataset": "data/keluhan_imbalanced.csv",
  "arsitektur": "meanpool",
  "embedding_dim": 100,
  "vocab": 1277,
  "epochs": 10,
  "device": "cuda",
  "n_train": 1207,
  "n_val": 214,
  "n_test": 356,
  "akurasi": 0.6938,
  "f1_macro": 0.5626,
  "f1_val_terbaik": 0.5001
}

muat ulang cocok: True


---
## Catatan khusus case imbalanced (PyTorch)

**Padanan `class_weight="balanced"` di PyTorch adalah `weight=` pada loss function.** Tidak ada
parameter ajaib di model — bobotnya kamu hitung sendiri dari frekuensi kelas di data latih:

```python
n = np.bincount(Y[idx_tr], minlength=N_OUT)
bobot = torch.tensor(len(idx_tr) / (N_OUT * n), dtype=torch.float, device=device)
criterion = nn.CrossEntropyLoss(weight=bobot)
```

Rumusnya sama persis dengan yang dipakai sklearn: `n_sampel / (n_kelas * jumlah_kelas_ini)`.
Kelas yang jarang dapat bobot besar, sehingga kesalahan padanya "lebih mahal" bagi loss.

- **Accuracy berbohong di sini.** §8 mencetak akurasi model yang selalu menebak kelas mayoritas
  sebagai pembanding — kalau modelmu tidak jauh dari angka itu, ia belum belajar apa-apa.
- Yang dilaporkan: **f1 / recall kelas minoritas**.
- Kalau bobot loss belum cukup, opsi lain: turunkan ambang keputusan (pakai
  `F.softmax(logits, 1)[:, i_min] > 0.3` alih-alih `argmax`), atau oversample kelas minoritas
  lewat `WeightedRandomSampler` di `DataLoader`.
- `stratify` sudah dipakai di §5 saat split — wajib untuk data timpang, kalau tidak kelas
  minoritas bisa nyaris tidak masuk test set.